## Scoring and Evaluation Metrics of the Content-based Recommender System

In [24]:
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.metrics import roc_auc_score
from typing import Tuple
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")

## Helper functions to load and evaluate the scoring

In [25]:
def load_evaluation_data() -> Tuple[pd.DataFrame, csr_matrix, pd.DataFrame]:
    dim_tracks = pd.read_csv('../../data/processed/to_db/dim_tracks.csv')
    dim_artists = pd.read_csv('../../data/processed/to_db/dim_artists.csv')
    fact_interactions = pd.read_csv('../../data/processed/to_db/fact_interactions.csv')

    all_artists = pd.read_csv('all_artists.csv').fillna({'tags': '', 'similar_artists': ''})
    combined_feats = load_npz('feature_matrix.npz')
    
    if 'artists_id' not in all_artists.columns:
        all_artists = all_artists.merge(
            dim_artists[['primary_artist_name', 'artists_id']], 
            on='primary_artist_name', 
            how='left'
        )
    
    all_artists['artists_id'] = all_artists['artists_id'].astype('Int64')
    
    interaction_weights = {'Long Top Track': 1.0, 'Saved': 0.8, 'Recently Played': 0.75}
    user_data = fact_interactions.merge(dim_tracks, how='left', on='track_id').merge(dim_artists, how='left', on='artists_id')
    user_data['weight'] = user_data['interaction_type'].map(interaction_weights).fillna(1.0)
    
    user_profile = user_data.groupby('primary_artist_name')['weight'].sum().reset_index()
    return user_profile, combined_feats, all_artists

def build_profile(X_mat, rows, w):
    p = np.asarray(X_mat[rows].multiply(np.asarray(w).reshape(-1, 1)).sum(axis=0)).ravel()
    return p / np.linalg.norm(p)

def full_scores_profile(profile):
    return np.asarray(X @ profile).ravel()

def full_scores_knn(library_rows, k=10):
    return np.sort(S[:, library_rows], axis=1)[:, -k:].mean(axis=1)


user_profile, feature_matrix, all_artists_df = load_evaluation_data()

In [26]:
X = feature_matrix
S = (X @ X.T).toarray()

dim_tracks = pd.read_csv('../../data/processed/to_db/dim_tracks.csv')
fact_interactions = pd.read_csv('../../data/processed/to_db/fact_interactions.csv')

is_lib = (all_artists_df['source'] == 'library').to_numpy()
lib_rows = np.where(is_lib)[0]
cand_rows = np.where(~is_lib)[0]
listeners = all_artists_df['artist_listeners'].to_numpy(dtype=float)

interactions = fact_interactions.merge(dim_tracks[['track_id', 'artists_id']], on='track_id', how='left')
weights = {'Long Top Track': 1.0, 'Saved': 0.8, 'Recently Played': 0.25}
interactions['weight'] = interactions['interaction_type'].map(weights).fillna(0.5)
signal = interactions.groupby('artists_id')['weight'].sum()

w_all = np.zeros(len(all_artists_df))
w_all[lib_rows] = 1 + np.log1p(all_artists_df.loc[is_lib, 'artists_id'].map(signal).fillna(0).to_numpy(dtype=float))

In [27]:
def holdout(n_seeds=10, frac=0.2, top_k=20):
    rows = []
    for seed in range(n_seeds):
        rng = np.random.default_rng(seed)
        test = rng.choice(lib_rows, size=int(frac * len(lib_rows)), replace=False)
        train = np.setdiff1d(lib_rows, test)
        pool = np.concatenate([cand_rows, test])
        y = np.r_[np.zeros(len(cand_rows)), np.ones(len(test))]
        
        scorers = {
            'popularity': listeners,
            'knn10': full_scores_knn(train),
            'profile (weighted)': full_scores_profile(build_profile(X, train, w_all[train]))
        }
        
        for name, s_all in scorers.items():
            s = np.asarray(s_all).ravel()[pool]
            top = np.argsort(-s)[:top_k]
            rows.append((seed, name, roc_auc_score(y, s), y[top].mean()))
            
    return pd.DataFrame(rows, columns=['seed', 'method', 'auc', f'precision@{top_k}']).groupby('method').agg(['mean', 'std']).round(3)

eval_results = holdout()
print(eval_results)

                   seed           auc        precision@20       
                   mean    std   mean    std         mean    std
method                                                          
knn10               4.5  3.028  0.679  0.029        0.380  0.098
popularity          4.5  3.028  0.751  0.022        0.445  0.112
profile (weighted)  4.5  3.028  0.664  0.018        0.395  0.060
